In [ ]:
import pandas as pd
import re
import gc

from tqdm import tqdm
from nltk.tokenize import TweetTokenizer
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

In [ ]:
tokenizer = TweetTokenizer()
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

In [ ]:
# Function to clean data   
def clean_tweet(tweet):
    # Remove special characters, numbers, links,username, hashtags, and punctuation
    cleaned_tweet = str(tweet)
    
    # text case normalization
    cleaned_tweet = cleaned_tweet.lower()
    
    # Remove usernames and mentions
    cleaned_tweet = re.sub(r'@[A-Za-z0-9_]+', '', cleaned_tweet)
    
    # Remove links 
    cleaned_tweet = re.sub(r'http[s]?://\S+', '', cleaned_tweet)
    
    # Remove hashtags 
    cleaned_tweet = re.sub(r'#\w+', '', cleaned_tweet)
    
    # Remove Characters
    cleaned_tweet = re.sub(r'[^a-zA-Z\s]', '', cleaned_tweet)
    
    # Remove extra spaces
    cleaned_tweet = re.sub(r'\s+', ' ', cleaned_tweet).strip()
    
    return cleaned_tweet

In [ ]:
def tokenize_tweet(tweet):
    # Tokenization
    tokens = tokenizer.tokenize(tweet)
    
    # Remove stopwords
    filtered_tokens = [word for word in tokens if word.lower() not in stop_words]
    
    # Lemmatization
    lemmatized_tokens = [lemmatizer.lemmatize(token) for token in filtered_tokens]
    
    return lemmatized_tokens

In [ ]:
def final_processing(filename):   
    # file = '../ProcessedData/' + filename 
    file = filename
    print('Reading', filename)
    data = pd.read_csv(file, lineterminator='\n')
    print(filename, 'Read')
    print('Cleaning', filename)
    data['cleaned_tweet'] = data['text'].apply(clean_tweet)
    print('Data Cleaned')
    print('Tokenizing', filename)
    data['tokenized_tweet'] = data['cleaned_tweet'].apply(tokenize_tweet)
    print('Data Tokenized')
    print('Dropping empty')
    data.dropna(subset=['tokenized_tweet'], inplace=True)
    print('Saving', filename)
    data.to_csv(file, index=False)
    print('Processed', filename)
    del data
    gc.collect()

In [ ]:
csv_files = ['Week1.csv','Week2.csv','Week3.csv']
for csv_file in tqdm(csv_files):
    final_processing(csv_file)

In [ ]:
from isort import file

def tweet_extract(csv_file, company):
    """ 
        Extract tweets related to companies from the processed data 
        Args: 
            csv_file (str): Name of the csv file to be processed
            company (str): Company name to extract tweets related to
        Returns:
            pd.DataFrame: Dataframe containing tweets related to the company
    """
    
    # file_path = '../ProcessedData/' + csv_file
    file_path = csv_file
    df_list = []
    for batch in pd.read_csv(file_path, chunksize=1000000, lineterminator='\n'):
        print('Read - ',batch.shape)
        
        # Query for extracting tweets related to companies
        query = [company]
        # batch['query_word'] = batch['cleaned_tweet'].apply(lambda text: all(re.search(r'\b{}\b'.format(re.escape(word)), text.lower()) for word in query))
        batch['query_word'] = batch['cleaned_tweet'].apply(lambda cleaned_tweet: all(word in str(cleaned_tweet).lower() for word in query))
        company_tweets = batch[batch['query_word']]
        
        # Drop duplicates
        print('droping duplicates')
        company_tweets = company_tweets.drop_duplicates(subset=['text'])
        
        # Adding extracted tweets to list
        df_list.append(company_tweets)
        print('Extracted tweets related to companies and Work experience - ',company_tweets.shape)
        
        del batch  # Delete batch to free up memory
        gc.collect()  # Garbage collect to free up memory
        
    print(csv_file, 'processed')
    return pd.concat(df_list, ignore_index=True)

In [ ]:
# Extract tweets related to companies
files = ['Final_Processed_data.csv']
companies = ['microsoft', 'google', 'apple']

for company in tqdm(companies):
    company_tweet = pd.DataFrame()
    for csv in files:
        extract = tweet_extract(csv, company)
        company_tweet = pd.concat([company_tweet, extract], ignore_index=True)
    print(company_tweet.shape)
    company_tweet.to_csv('../ProcessedData/sept2020'+company+'_tweets.csv', index=False)
    print(company, 'processed')
    del company_tweet